In [1]:
# =========================================
# 1. Import
# =========================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================================
# 2. Load Data
# =========================================
df = pd.read_csv("revise_Chinese_resume_data.csv")

In [2]:
# =========================================
# 3. Data Cleaning
# =========================================
df["label"] = df["筛选结果"].map({"通过": 1, "不通过": 0})
df = df.drop(columns=["筛选结果"])
df = df.fillna(0)

# =========================================
# 4. 特征工程
# =========================================

feature_df = pd.DataFrame()


feature_df["年龄"] = df["年龄"].astype(float)
feature_df["小规模项目"] = df["小规模项目"].astype(float)
feature_df["中规模项目"] = df["中规模项目"].astype(float)
feature_df["大规模项目"] = df["大规模项目"].astype(float)


feature_df = pd.concat([feature_df, pd.get_dummies(df["学历层次"], prefix="学历")], axis=1)
feature_df = pd.concat([feature_df, pd.get_dummies(df["院校类别"], prefix="院校")], axis=1)
feature_df = pd.concat([feature_df, pd.get_dummies(df["英语水平"], prefix="英语")], axis=1)


exp_features = ["小型企业工作经验", "中型企业工作经验", "大型企业工作经验"]
for col in exp_features:
    feature_df[col + "_有经验"] = (df[col] != "无").astype(int)


proficiency_cols = [
    "编程语言熟练度", "前端技术熟练度", "后端技术熟练度",
    "数据库熟练度", "云计算/运维熟练度", "数据与算法熟练度",
    "移动开发熟练度", "测试工具熟练度"
]

for col in proficiency_cols:
    feature_df[col + "_count"] = df[col].apply(
        lambda x: len(str(x).split(",")) if pd.notna(x) and str(x).strip() and x != 0 else 0
    )


feature_df = pd.concat([feature_df, pd.get_dummies(df["意向岗位"], prefix="岗位")], axis=1)


feature_df["性别_男"] = (df["性别"] == "男").astype(int)
feature_df["专业_计算机类"] = (df["专业类别"] == "计算机类").astype(int)

# =========================================
# 5. 准备训练数据
# =========================================
X = feature_df
y = df["label"]

print("=" * 60)
print("简化特征工程总结")
print("=" * 60)
print(f"总特征数: {X.shape[1]}")
print(f"\n特征类型分布:")
print(f"  - 数值特征: 4 (年龄, 项目数)")
print(f"  - 学历One-Hot: 3")
print(f"  - 院校One-Hot: 3")
print(f"  - 英语One-Hot: 3")
print(f"  - 工作经验二值: 3")
print(f"  - 技能数量: 8")
print(f"  - 岗位One-Hot: 10")
print(f"  - 其他: 2 (性别, 专业)")

简化特征工程总结
总特征数: 36

特征类型分布:
  - 数值特征: 4 (年龄, 项目数)
  - 学历One-Hot: 3
  - 院校One-Hot: 3
  - 英语One-Hot: 3
  - 工作经验二值: 3
  - 技能数量: 8
  - 岗位One-Hot: 10
  - 其他: 2 (性别, 专业)


In [3]:
from sklearn.tree import DecisionTreeClassifier

# =========================================
# 6. Train/Test Split
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# 7. Train Decision Tree 
# =========================================
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# =========================================
# 8. Evaluation
# =========================================
print("=" * 60)
print("Baseline Model (决策树 max_depth=3)")
print("=" * 60)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")

# =========================================
# 9. Feature Importance
# =========================================
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\n" + "=" * 60)
print("决策树特征重要性 (Top 15)")
print("=" * 60)
print(feature_importance.head(15).to_string(index=False))

# =========================================
# 10. 决策树可视化
# =========================================
from sklearn import tree
print("\n" + "=" * 60)
print("决策树结构")
print("=" * 60)
print(tree.export_text(model, feature_names=list(X.columns), max_depth=3))

Baseline Model (决策树 max_depth=3)
Accuracy : 0.7450
Precision: 0.8507
Recall   : 0.5992
F1 Score : 0.7031

决策树特征重要性 (Top 15)
       feature  importance
  数据库熟练度_count    0.417343
            年龄    0.203406
数据与算法熟练度_count    0.178450
         学历_专科    0.141337
         小规模项目    0.059465
         中规模项目    0.000000
         大规模项目    0.000000
      学历_硕士及以上    0.000000
      院校_985高校    0.000000
       院校_普通高校    0.000000
          英语_无    0.000000
         学历_本科    0.000000
       英语_英语六级    0.000000
       英语_英语四级    0.000000
  中型企业工作经验_有经验    0.000000

决策树结构
|--- 数据库熟练度_count <= 0.50
|   |--- 数据与算法熟练度_count <= 0.50
|   |   |--- 年龄 <= 22.50
|   |   |   |--- class: 0
|   |   |--- 年龄 >  22.50
|   |   |   |--- class: 0
|   |--- 数据与算法熟练度_count >  0.50
|   |   |--- 年龄 <= 22.50
|   |   |   |--- class: 0
|   |   |--- 年龄 >  22.50
|   |   |   |--- class: 1
|--- 数据库熟练度_count >  0.50
|   |--- 学历_专科 <= 0.50
|   |   |--- 年龄 <= 22.50
|   |   |   |--- class: 1
|   |   |--- 年龄 >  22.50
|   |   |   |--- c

In [4]:
# 完整特征重要性表
print(feature_importance.to_markdown())

|    | feature                 |   importance |
|---:|:------------------------|-------------:|
| 19 | 数据库熟练度_count      |    0.417343  |
|  0 | 年龄                    |    0.203406  |
| 21 | 数据与算法熟练度_count  |    0.17845   |
|  4 | 学历_专科               |    0.141337  |
|  1 | 小规模项目              |    0.0594653 |
|  2 | 中规模项目              |    0         |
|  3 | 大规模项目              |    0         |
|  6 | 学历_硕士及以上         |    0         |
|  8 | 院校_985高校            |    0         |
|  9 | 院校_普通高校           |    0         |
| 10 | 英语_无                 |    0         |
|  5 | 学历_本科               |    0         |
| 11 | 英语_英语六级           |    0         |
| 12 | 英语_英语四级           |    0         |
| 14 | 中型企业工作经验_有经验 |    0         |
| 13 | 小型企业工作经验_有经验 |    0         |
| 15 | 大型企业工作经验_有经验 |    0         |
| 16 | 编程语言熟练度_count    |    0         |
| 17 | 前端技术熟练度_count    |    0         |
|  7 | 院校_211高校            |    0         |
| 18 | 后端技术熟练度_count    |    0         |
| 20 | 云计算/运维熟练度_count | 

In [5]:
from sklearn.metrics import classification_report, confusion_matrix

# =========================================
# 11. Confusion Matrix
# =========================================
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(f"              预测不通过  预测通过")
print(f"实际不通过      {cm[0,0]:5d}      {cm[0,1]:5d}")
print(f"实际通过        {cm[1,0]:5d}      {cm[1,1]:5d}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================================
# 12. 加载LLM评估结果
# =========================================
print("\n" + "=" * 60)
print("加载LLM评估结果")
print("=" * 60)

llm_df = pd.read_csv("resume_screening_results.csv", encoding="utf-8")
llm_df["llm_score"] = pd.to_numeric(llm_df["模型筛选结果"], errors="coerce").fillna(0.5)

print(f"LLM评分分布:")
print(llm_df["llm_score"].value_counts().sort_index())

# 添加LLM评分到特征矩阵
X["llm_score"] = llm_df["llm_score"].values

# =========================================
# 13. Baseline vs Enhanced 模型对比
# =========================================
print("\n" + "=" * 60)
print("模型对比: Baseline vs Enhanced (含LLM特征)")
print("=" * 60)

# Baseline (无LLM)
X_base = X.drop(columns=["llm_score"])
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

model_b = DecisionTreeClassifier(max_depth=3, random_state=42)
model_b.fit(X_train_b, y_train_b)
y_pred_b = model_b.predict(X_test_b)

# Enhanced (含LLM)
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_e = DecisionTreeClassifier(max_depth=3, random_state=42)
model_e.fit(X_train_e, y_train_e)
y_pred_e = model_e.predict(X_test_e)

# 对比输出
print("\n--- Baseline (无LLM) ---")
print(f"Accuracy : {accuracy_score(y_test_b, y_pred_b):.4f}")
print(f"Precision: {precision_score(y_test_b, y_pred_b):.4f}")
print(f"Recall   : {recall_score(y_test_b, y_pred_b):.4f}")
print(f"F1 Score : {f1_score(y_test_b, y_pred_b):.4f}")

print("\n--- Enhanced (含LLM) ---")
print(f"Accuracy : {accuracy_score(y_test_e, y_pred_e):.4f}")
print(f"Precision: {precision_score(y_test_e, y_pred_e):.4f}")
print(f"Recall   : {recall_score(y_test_e, y_pred_e):.4f}")
print(f"F1 Score : {f1_score(y_test_e, y_pred_e):.4f}")

print("\n--- Performance Gain ---")
acc_gain = (accuracy_score(y_test_e, y_pred_e) - accuracy_score(y_test_b, y_pred_b)) * 100
prec_gain = (precision_score(y_test_e, y_pred_e) - precision_score(y_test_b, y_pred_b)) * 100
rec_gain = (recall_score(y_test_e, y_pred_e) - recall_score(y_test_b, y_pred_b)) * 100
f1_gain = (f1_score(y_test_e, y_pred_e) - f1_score(y_test_b, y_pred_b)) * 100

print(f"Accuracy : {'+' if acc_gain >= 0 else ''}{acc_gain:.2f}%")
print(f"Precision: {'+' if prec_gain >= 0 else ''}{prec_gain:.2f}%")
print(f"Recall   : {'+' if rec_gain >= 0 else ''}{rec_gain:.2f}%")
print(f"F1 Score : {'+' if f1_gain >= 0 else ''}{f1_gain:.2f}%")

# =========================================
# 14. LLM特征重要性分析
# =========================================
print("\n" + "=" * 60)
print("LLM特征重要性分析")
print("=" * 60)

feature_importance_e = pd.DataFrame({
    "feature": X.columns,
    "importance": model_e.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nEnhanced模型 Top 15 特征:")
print(feature_importance_e.head(15).to_string(index=False))

# LLM特征重要性
llm_row = feature_importance_e[feature_importance_e["feature"] == "llm_score"]
if len(llm_row) > 0:
    llm_imp = llm_row["importance"].values[0]
    llm_rank = feature_importance_e.reset_index(drop=True)
    llm_rank = llm_rank[llm_rank["feature"] == "llm_score"].index[0] + 1
    print(f"\nLLM Score 重要性: {llm_imp:.4f}")
    print(f"LLM Score 重要性排名: {llm_rank}/{len(X.columns)}")

# =========================================
# 15. Confusion Matrix 对比
# =========================================
print("\n" + "=" * 60)
print("Confusion Matrix 对比")
print("=" * 60)

cm_b = confusion_matrix(y_test_b, y_pred_b)
cm_e = confusion_matrix(y_test_e, y_pred_e)

print("\nBaseline (无LLM):")
print(f"              预测不通过  预测通过")
print(f"实际不通过      {cm_b[0,0]:5d}      {cm_b[0,1]:5d}")
print(f"实际通过        {cm_b[1,0]:5d}      {cm_b[1,1]:5d}")

print("\nEnhanced (含LLM):")
print(f"              预测不通过  预测通过")
print(f"实际不通过      {cm_e[0,0]:5d}      {cm_e[0,1]:5d}")
print(f"实际通过        {cm_e[1,0]:5d}      {cm_e[1,1]:5d}")

print("\n--- Baseline ---")
print(classification_report(y_test_b, y_pred_b))

print("\n--- Enhanced ---")
print(classification_report(y_test_e, y_pred_e))

Confusion Matrix:
              预测不通过  预测通过
实际不通过        443         53
实际通过          202        302

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.89      0.78       496
           1       0.85      0.60      0.70       504

    accuracy                           0.74      1000
   macro avg       0.77      0.75      0.74      1000
weighted avg       0.77      0.74      0.74      1000


加载LLM评估结果
LLM评分分布:
llm_score
0    2955
1    2045
Name: count, dtype: int64

模型对比: Baseline vs Enhanced (含LLM特征)

--- Baseline (无LLM) ---
Accuracy : 0.7450
Precision: 0.8507
Recall   : 0.5992
F1 Score : 0.7031

--- Enhanced (含LLM) ---
Accuracy : 0.7880
Precision: 0.8333
Recall   : 0.7242
F1 Score : 0.7749

--- Performance Gain ---
Accuracy : +4.30%
Precision: -1.74%
Recall   : +12.50%
F1 Score : +7.18%

LLM特征重要性分析

Enhanced模型 Top 15 特征:
        feature  importance
      llm_score    0.758281
  后端技术熟练度_count    0.086357
          小规模项目    0.05